<a href="https://colab.research.google.com/github/Zinniacodes01/Zinnia-s-Code/blob/main/text_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#imports
import string
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
file=open("spa.txt","r")
text=file.read()

In [ ]:
english_list=[]
spanish_list=[]
def split_text(text):
  line_by_line=text.split('\n')
  for item in line_by_line:
    split_text=item.split('\t')
    if len(split_text)==2:
      english_list.append(split_text[0])
      spanish_list.append(split_text[1])
split_text(text)

In [ ]:
english=np.array(english_list)
spanish=np.array(spanish_list)

In [ ]:
train=tf.data.Dataset.from_tensor_slices((spanish,english)).shuffle(buffer_size=len(english)).batch(64)

In [ ]:
import tensorflow_text as tf_text
def clean_text(text):
  #removes accent
  text=tf_text.normalize_utf8(text,'NFKD')
  #lowercase
  text=tf.strings.lower(text)
  #keep specific punc.
  text=tf.strings.regex_replace(text,'[^ a-z.?!,]','')# ^ means everything except
  #add spaces after punctuation????
  text=tf.strings.regex_replace(text,'[.?!,]',r' \0')
  #strip(remove extra spaces at beginning + end of string)
  text=tf.strings.strip(text)
  #wrap every phrase with start and end token
  text=tf.strings.join(['[START]',text,'[END]'],separator=' ')
  return text
clean_text('¡Ven rápido!')

In [ ]:
#text vectorization layer
max_features=5000
spanish_text_vectorize=tf.keras.layers.TextVectorization(
    max_tokens=max_features,
    standardize=clean_text,
    ragged=True #all text is of different size
)
english_text_vectorize=tf.keras.layers.TextVectorization(
    max_tokens=max_features,
    standardize=clean_text,
    ragged=True
)

In [ ]:
spanish_text_vectorize.adapt(train.map(lambda spanish, english: spanish))
english_text_vectorize.adapt(train.map(lambda spanish, english: english))

In [ ]:
spanish_text_vectorize.get_vocabulary()[:100]

['',
 '[UNK]',
 np.str_('[START]'),
 np.str_('[END]'),
 np.str_('.'),
 np.str_('que'),
 np.str_('de'),
 np.str_('el'),
 np.str_('a'),
 np.str_('no'),
 np.str_('tom'),
 np.str_('la'),
 np.str_('?'),
 np.str_('en'),
 np.str_('es'),
 np.str_('un'),
 np.str_('se'),
 np.str_('me'),
 np.str_(','),
 np.str_('esta'),
 np.str_('por'),
 np.str_('lo'),
 np.str_('una'),
 np.str_('mi'),
 np.str_('su'),
 np.str_('los'),
 np.str_('con'),
 np.str_('le'),
 np.str_('ella'),
 np.str_('te'),
 np.str_('para'),
 np.str_('mary'),
 np.str_('y'),
 np.str_('las'),
 np.str_('mas'),
 np.str_('tu'),
 np.str_('al'),
 np.str_('como'),
 np.str_('yo'),
 np.str_('este'),
 np.str_('estoy'),
 np.str_('muy'),
 np.str_('eso'),
 np.str_('tiene'),
 np.str_('si'),
 np.str_('del'),
 np.str_('estaba'),
 np.str_('quiero'),
 np.str_('tengo'),
 np.str_('fue'),
 np.str_('aqui'),
 np.str_('casa'),
 np.str_('cuando'),
 np.str_('hacer'),
 np.str_('puedo'),
 np.str_('todo'),
 np.str_('algo'),
 np.str_('esto'),
 np.str_('hay'),
 np.str_

In [ ]:
#preprocess
#neural macine translation: with attention
#context=spanish
#target=english
def text_process(context, target):
  context=spanish_text_vectorize(context).to_tensor()#pad w ith zeroes
  target=english_text_vectorize(target)
  target_input=target[:,:-1].to_tensor()#keeping everything except the last token
  target_output=target[:,1:].to_tensor()
  return (context, target_input), target_output

In [ ]:
train_ds=train.map(text_process,tf.data.AUTOTUNE)

In [ ]:
class Encoder(tf.keras.layers.Layer):
  def __init__(self, text_processor, units):
    super(Encoder,self).__init__()
    self.processor=text_processor
    self.units=units
    self.vocab_size=text_processor.vocabulary_size()
    self.embedding=tf.keras.layers.Embedding(self.vocab_size,self.units,mask_zero=True)#convert text to #s(vectors)
    self.rnn=tf.keras.layers.Bidirectional(merge_mode='sum',layer=tf.keras.layers.GRU(self.units,return_sequences=True,recurrent_initializer='glorot_uniform'))
    #learns from both directions(diff. b/w the cat sat and the sat cat)
  def call(self,x):
    x=self.embedding(x)
    x=self.rnn(x)
    return x
  def convert_input(self,x):
    text=tf.convert_to_tensor(x)
    if len(text.shape)==0:
      text=tf.convert_to_tensor(text)[tf.newaxis]#if shape isnt 2d then we make it 2d
    context=self.processor(text).to_tensor()
    context=self(context)#calling the function
    return context

In [ ]:
#create encoder
units=256
encoder=Encoder(spanish_text_vectorize,units)

In [ ]:
class Attention(tf.keras.layers.Layer):
  def __init__(self,units,**kwargs):
    #other keyword agruments passed along used as is
    super().__init__()
    self.mha=tf.keras.layers.MultiHeadAttention(num_heads=1,key_dim=units,**kwargs)
    self.normalization=tf.keras.layers.LayerNormalization()
    self.add=tf.keras.layers.Add()
  def call(self,x,context):
    attention_output,attention_scores=self.mha(query=x,value=context,return_attention_scores=True)#query=read and value=entire input
    self.last_attention_weight=tf.reduce_mean(attention_scores,axis=1)
    x=self.add([x,attention_output])
    x=self.normalization(x)
    return x

In [ ]:
#attention_layer=Attention(units)
#wont be used anywhere

In [ ]:
class Decoder(tf.keras.layers.Layer):
  def __init__(self, text_processor, units):
    super(Decoder, self).__init__()
    self.text_processor=text_processor
    self.units=units
    self.vocab_size=text_processor.vocabulary_size()
    self.word_to_id=tf.keras.layers.StringLookup(vocabulary=text_processor.get_vocabulary(),mask_token="",oov_token='[UNK]')#oov=out of vocabulary
    self.id_to_word=tf.keras.layers.StringLookup(vocabulary=text_processor.get_vocabulary(),mask_token="",oov_token='[UNK]',invert=True)
    self.start_token=self.word_to_id('[START]')
    self.end_token=self.word_to_id('[END]')
    #layers
    self.embedding=tf.keras.layers.Embedding(self.vocab_size,self.units,mask_zero=True)
    self.rnn=tf.keras.layers.GRU(self.units,return_sequences=True,return_state=True,recurrent_initializer='glorot_uniform')
    self.attention_layer=Attention(units)
    self.text_output=tf.keras.layers.Dense(self.vocab_size)
  def call(self,x,context,state=None,return_state=False):
    x=self.embedding(x)
    x,state=self.rnn(x,initial_state=state)
    x=self.attention_layer(x,context)
    self.attention_weight=self.attention_layer.last_attention_weight
    x=self.text_output(x)
    if return_state==True:
      return x, state
    else:
      return x
  def get_initial_state(self,context):
    batch_size=tf.shape(context)[0]
    start_tokens=tf.fill([batch_size,1],self.start_token)
    done=tf.zeros([batch_size,1],dtype=tf.bool)
    return start_tokens,done,self.rnn.get_initial_state(batch_size=batch_size)[0]
  def get_next_token(self,context,next_token,done,state,temperature=0):
    logits,state=self(next_token,context,state=state,return_state=True)
    if temperature==0:
      next_token=tf.argmax(logits,axis=-1)#highest probability category
    else:
      logits=logits[:,-1,:]/temperature
      next_token=tf.random.categorical(logits,num_samples=1)
    done=done|(next_token==self.end_token)
    next_token=tf.where(done,tf.constant(0, dtype=tf.int64),next_token)
    return next_token,done,state
  def tokens_to_text(self,tokens):
    words=self.id_to_word(tokens)
    sentence=tf.strings.reduce_join(words,axis=-1,separator=' ')
    sentence_V2=tf.strings.regex_replace(sentence,r'^ *\[START\] *','')#^
    final_sentence=tf.strings.regex_replace(sentence,r' *\[END\] *$','')#$
    return final_sentence

translator class: everything comes together

In [ ]:
class Translator(tf.keras.Model):
  def __init__(self,units,spanish_layer,english_layer):
    super().__init__()
    self.encoder=Encoder(spanish_layer,units)
    self.decoder=Decoder(english_layer,units)
  def call(self,inputs):
    context,x=inputs
    context=self.encoder(context)
    logits=self.decoder(x,context)
    return logits
  def translate(self,texts,*kwargs,max_length=50,temperature=0):
    tokens=[]
    attention_weights=[]
    context=self.encoder.convert_input(texts)
    batch_size=tf.shape(texts)[0]
    next_token,done,state=self.decoder.get_initial_state(context)
    for n in range(max_length):
      next_token,done,state=self.decoder.get_next_token(context,next_token,done,state,temperature=0)
      tokens.append(next_token)
      attention_weights.append(self.decoder.attention_weight)
      if tf.executing_eagerly() and tf.reduce_all(done):
        break
    tokens=tf.concat(tokens,axis=-1)
    result=self.decoder.tokens_to_text(tokens)
    return result

translator model

In [ ]:
translator_model=Translator(units,spanish_text_vectorize,english_text_vectorize)

In [ ]:
def masked_loss(y_true,y_predict):
  loss_function=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True,reduction="none")
  loss=loss_function(y_true,y_predict)
  mask=tf.cast(y_true!=0,loss.dtype)
  loss*=mask
  return tf.reduce_sum(loss)/tf.reduce_sum(mask)
def masked_accuracy(y_true,y_predict):
  y_predict=tf.argmax(y_predict,axis=-1)
  y_predict=tf.cast(y_predict,y_true.dtype)
  match_value=tf.cast(y_true==y_predict,dtype=tf.float32)
  mask=tf.cast(y_true!=0,tf.float32)
  return tf.reduce_sum(match_value)/tf.reduce_sum(mask)

In [ ]:
translator_model.compile(optimizer='Adam',loss=masked_loss,metrics=[masked_loss,masked_accuracy])

In [ ]:
translator_model.fit(train_ds,epochs=20,callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'decoder_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'attention_1' (of type Attention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


1081/1859 ━━━━━━━━━━━━━━━━━━━━ 23s 31ms/step - loss: 3.5688 - masked_accuracy: 0.4431 - masked_loss: 3.5688

In [ ]:
#predictions
result=translator_model.translate([''])
print(result[0].numpy().decode())
#¿Puede hablar más despacio, por favor?-works
#¿Cuándo es tu cumpleaños?-works